# 02 · Baselines: keyword rules and the byte-level CNN

**CPU is fine (≈ 10–20 min); a GPU makes it a few minutes.**

* **Rules**: regular expressions for common injection wording (baseline 1).
* **Char-CNN**: a byte-level CNN (course week 4). Because it has no sub-word tokeniser, it is
  hardening method 3: it may be naturally more robust to character tricks.
* **Char-CNN + augmentation**: the same model trained with random obfuscation (method 2).

Every detector is scored on the whole bundle twice: on the raw input and after the normaliser
(method 1). Three seeds per trained model.

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
# os.environ["PI_SMOKE"] = "1"   # uncomment for a 5-minute synthetic test of the whole pipeline

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")

from pi_hardening import config
print("smoke mode:", config.SMOKE, "| data:", config.DATA_DIR)

smoke mode: False | data: C:\Users\admin\PycharmProjects\prompt-injection-hardening\data\processed


In [2]:
from dataclasses import asdict
from pi_hardening import data, evaluate
from pi_hardening.detectors.rules import RuleDetector
from pi_hardening.detectors.char_cnn import train_char_cnn
from pi_hardening.normalize import Normalizer
from pi_hardening.utils import describe_hardware
print(describe_hardware())
splits = data.load_splits()
bundle = evaluate.load_bundle()
norm = Normalizer()
latency_texts = splits["test_in"]["text"].tolist()
latency = []

CUDA: NVIDIA GeForce RTX 4070 SUPER, 12.0 GB, torch 2.11.0+cu128


## Rules

In [3]:
rules = RuleDetector()
preds = evaluate.score_bundle(rules, bundle, model="rules")
evaluate.save_predictions(preds, "rules")
latency += [evaluate.measure_latency(rules, latency_texts, "rules"),
            evaluate.measure_latency(rules, latency_texts, "rules", normaliser=norm)]
evaluate.metrics_table(preds).query("eval_set == 'test_in'")[["condition", "family", "tpr", "fpr", "f1"]]

,condition,family,tpr,fpr,f1
1,norm,char_spacing,0.406,0.000,0.578
2,norm,clean,0.583,0.000,0.736
3,norm,diacritics,0.583,0.000,0.736
4,norm,homoglyph,0.529,0.000,0.692
5,norm,leetspeak,0.567,0.000,0.724
6,norm,punct_split,0.583,0.000,0.736
7,norm,zero_width,0.583,0.000,0.736
16,raw,char_spacing,0.219,0.000,0.360
17,raw,clean,0.583,0.000,0.736
18,raw,diacritics,0.214,0.000,0.352


## Char-CNN (plain and with obfuscation augmentation)

In [4]:
cfg = config.CharCNNConfig()
aug = config.AugmentConfig()
print(asdict(cfg)); print(asdict(aug))
for model_name, augment in [("char_cnn", None), ("char_cnn_aug", aug)]:
    for seed in cfg.seeds:
        print(f"{model_name}  seed {seed}")
        det, hist = train_char_cnn(splits["train"], splits["val"], cfg, seed=seed, augment=augment, name=model_name)
        evaluate.save_history(hist, model_name, seed)
        evaluate.save_predictions(evaluate.score_bundle(det, bundle, model=model_name, seed=seed),
                                  f"{model_name}__s{seed}")
        if seed == cfg.seeds[0]:
            latency += [evaluate.measure_latency(det, latency_texts, model_name),
                        evaluate.measure_latency(det, latency_texts, model_name, normaliser=norm)]
        det.unload()
evaluate.save_latency(latency)

{'max_len': 512, 'emb_dim': 64, 'n_filters': 128, 'kernel_sizes': (3, 5, 7), 'dropout': 0.3, 'batch_size': 64, 'lr': 0.001, 'epochs': 15, 'seeds': (13, 42, 77)}
{'p': 0.5, 'rate_range': (0.2, 1.0), 'families': ['zero_width', 'homoglyph', 'diacritics', 'leetspeak', 'char_spacing', 'punct_split']}
char_cnn  seed 13


  epoch 1: loss 0.3655  val F1 0.9324
  epoch 2: loss 0.1694  val F1 0.9534


  epoch 3: loss 0.1255  val F1 0.9571
  epoch 4: loss 0.0826  val F1 0.9682


  epoch 5: loss 0.0676  val F1 0.9682
  epoch 6: loss 0.0524  val F1 0.9684


  epoch 7: loss 0.0374  val F1 0.9722
  epoch 8: loss 0.0351  val F1 0.9684


...

  epoch 11: loss 0.0628  val F1 0.9758


  epoch 12: loss 0.0521  val F1 0.9758


  epoch 13: loss 0.0436  val F1 0.9724


  epoch 14: loss 0.0470  val F1 0.9758


  epoch 15: loss 0.0412  val F1 0.9724


In [5]:
m = evaluate.metrics_table(evaluate.load_predictions("char_cnn*"))
m.query("eval_set == 'test_in'").pivot_table(index=["model", "condition"], columns="family", values="tpr")

family                  char_spacing  clean  diacritics  homoglyph  leetspeak  \
model        condition                                                          
char_cnn     norm              0.916  0.916       0.916      0.918      0.914   
             raw               0.697  0.916       0.938      0.888      0.906   
char_cnn_aug norm              0.902  0.900       0.900      0.900      0.900   
             raw               0.872  0.900       0.902      0.893      0.897   

family                  punct_split  zero_width  
model        condition                           
char_cnn     norm             0.916       0.916  
             raw              0.893       0.925  
char_cnn_aug norm             0.900       0.900  
             raw              0.904       0.897